# 14 - Overlay: BUY/SELL signals on price (AUTO)

Auto-picks the symbols that actually **have signals in the window** (most
signals first) and, for each, draws the price line with green BUY / red SELL
markers on the action dates. No typing needed.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# window comes from update_data.py - the SAME toggle as live vs backtest
from update_data import START_DATE, END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError('prices.parquet not found - run  python pull_bloomberg_prices.py  first.')
    px = pd.read_parquet(PRICES_PATH); px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    one = prices[prices['symbol'] == symbol].sort_values('date')
    return clip_series(one.set_index('date')['px_last'])


In [ ]:
HOW_MANY = 6      # how many of the most-signalled symbols to plot

In [ ]:
tick = pd.read_parquet(os.path.join(P, 'trade_signals_tickers.parquet'))
theme = pd.read_parquet(os.path.join(P, 'trade_signals.parquet'))
tick['action_date'] = pd.to_datetime(tick['action_date'])
theme['action_date'] = pd.to_datetime(theme['action_date'])
tick = clip_dates(tick, 'action_date')
theme = clip_dates(theme, 'action_date')

# AUTO-SELECT: build (symbol, action_date, action) rows from both signal files,
# then take the symbols with the most signals.
rows = []
for _, r in tick.iterrows():
    rows.append((r['ticker'], r['action_date'], r['action']))
for _, r in theme.iterrows():
    rows.append((r['etf'], r['action_date'], r['action']))
sig_all = pd.DataFrame(rows, columns=['symbol', 'action_date', 'action'])
symbols = sig_all['symbol'].value_counts().head(HOW_MANY).index.tolist()
print('auto-selected signalled symbols:', symbols)

prices = load_prices()
for symbol in symbols:
    px = price_series(prices, symbol)
    if px.empty:
        print('skip', symbol, '- no price rows'); continue
    s = sig_all[sig_all['symbol'] == symbol].copy()
    px_on_day = px.reindex(px.index.union(pd.to_datetime(s['action_date']))).sort_index().ffill()
    s['price'] = s['action_date'].map(px_on_day)
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(px.index, px.values, color='black', linewidth=1.2, label='price')
    buys = s[s['action'] == 'BUY']; sells = s[s['action'] == 'SELL']
    ax.scatter(buys['action_date'], buys['price'], marker='^', s=90, color='tab:green', zorder=5, label='BUY')
    ax.scatter(sells['action_date'], sells['price'], marker='v', s=90, color='tab:red', zorder=5, label='SELL')
    ax.set_ylabel('price (USD)'); ax.set_title(f'{symbol}: price with BUY/SELL signals')
    ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout(); plt.show()